# Run FF-SIMS HERA/ARES Fit

This notebook runs `lenscluster.cluster_solver` for either HERA or ARES using the active non-default settings from `run.xsh`. It detects the number of CPU cores available to this process and uses that value for both `JAX_NUM_CPU_DEVICES` and `--chains`.

Set `CLUSTER` to `"HERA"` or `"ARES"`, inspect the printed command, then execute the final cell.

In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import time
from pathlib import Path

PYTHON = "/home/dutra/.conda/envs/lenstronomy/bin/python"
REPO_ROOT = Path.cwd()

# Choose one: "HERA" or "ARES".
CLUSTER = "HERA"


def available_cpu_cores() -> int:
    try:
        return len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        return os.cpu_count() or 1


cores = available_cpu_cores()
print(f"Using {cores} CPU cores")

In [ ]:
output_dir_label = "jun22h_median_nostage0e1e2_rcoreff_mag22_scatter_arespriors_ellipticity_faster"

FF_RGB_DISPLAY = {
    "q": 6.8,
    "stretch": 0.0158,
    "minimum": 0.00105,
    "red_gain": 0.62,
    "green_gain": 0.78,
    "blue_gain": 3.65,
}

FF_RGB_BANDS = ["F435W", "F606W", "F814W"]

CLUSTER_CONFIGS = {
    "ARES": {
        "cluster_key": "ares",
        "par_path": "data/ff_sims/ares/ares_lenscluster.par",
        "output_dir": f"results/{output_dir_label}/ares",
        "kappa_true_fits": "data/ff_sims/published/ares/kappa_z9_0.fits",
        "gammax_true_fits": "data/ff_sims/published/ares/gammax_z9_0.fits",
        "gammay_true_fits": "data/ff_sims/published/ares/gammay_z9_0.fits",
    },
    "HERA": {
        "cluster_key": "hera",
        "par_path": "data/ff_sims/hera/hera_lenscluster.par",
        "output_dir": f"results/{output_dir_label}/hera",
        "kappa_true_fits": "data/ff_sims/published/hera/kappa_z9_0.fits",
        "gammax_true_fits": "data/ff_sims/published/hera/gammax_z9_0.fits",
        "gammay_true_fits": "data/ff_sims/published/hera/gammay_z9_0.fits",
        # HERA uses H0=72 km/s/Mpc, so h=0.72 and 2.3 h^-1 kpc = 3.19 kpc physical.
        "softening_length_kpc": 2.3 / 0.72,
        "softening_length_prior_log_sigma": 0.15,
    },
}

cluster = CLUSTER.strip().upper()
if cluster not in CLUSTER_CONFIGS:
    raise ValueError(f"Unknown CLUSTER={CLUSTER!r}; expected one of {sorted(CLUSTER_CONFIGS)}")

config = CLUSTER_CONFIGS[cluster]
par_path = Path(config["par_path"])
if not par_path.is_file():
    raise FileNotFoundError(f"Missing par file: {par_path}")

run_name = f"{config['cluster_key']}_S1source_S2none"
print(f"Cluster: {cluster}")
print(f"Par file: {par_path}")
print(f"Run name: {run_name}")

In [ ]:
# Non-default controls mirrored from run.xsh.
perturbation_discovery_alpha_tol_arcsec = 0.2
perturbation_discovery_jacobian_tol = 0.5
perturbation_discovery_jacobian_weight = 1.0

warmup = 3000
samples_for_output_name = 500  # Solver default; kept only to mirror the run.xsh output suffix.
max_tree_depth = 8
svi_steps = [10000, 10000]
refresh_every = [None, 1000]

independent_scaling_free_log_sigma_tau_prior_median = 0.45
independent_scaling_free_log_mass_tau_prior_median = 0.55
independent_scaling_free_log_tau_prior_sigma = 0.30

output_dir = (
    f"{config['output_dir']}_PD{perturbation_discovery_alpha_tol_arcsec:g}_"
    f"{perturbation_discovery_jacobian_tol:g}_T{max_tree_depth}W{warmup}S{samples_for_output_name}"
)
print(f"Output dir: {output_dir}")

In [ ]:
def extend_flag(args: list[str], flag: str, values) -> None:
    args.append(flag)
    if isinstance(values, (list, tuple)):
        args.extend("None" if value is None else str(value) for value in values)
    else:
        args.append("None" if values is None else str(values))


cmd = [
    PYTHON,
    "-m",
    "lenscluster.cluster_solver",
    "--resume",
    "all",
    "--par-path",
    str(par_path),
    "--output-dir",
    output_dir,
    "--run-name",
    run_name,
    "--best-value",
    "maximum-likelihood",
    "--potfile-member-mag-max",
    "22.0",
    "--stage1-likelihood",
    "source",
    "--warmup",
    str(warmup),
]
extend_flag(cmd, "--refresh-every", refresh_every)
extend_flag(cmd, "--svi-steps", svi_steps)
cmd.extend([
    "--chains",
    str(cores),
    "--target-accept",
    "0.8",
    "--max-tree-depth",
    str(max_tree_depth),
    "--z-bin-efficiency-tol",
    "0.0",
    "--perturbation-discovery-alpha-tol-arcsec",
    str(perturbation_discovery_alpha_tol_arcsec),
    "--perturbation-discovery-jacobian-tol",
    str(perturbation_discovery_jacobian_tol),
    "--perturbation-discovery-jacobian-weight",
    str(perturbation_discovery_jacobian_weight),
    "--independent-scaling-free-log-sigma-tau-prior-median",
    str(independent_scaling_free_log_sigma_tau_prior_median),
    "--independent-scaling-free-log-mass-tau-prior-median",
    str(independent_scaling_free_log_mass_tau_prior_median),
    "--independent-scaling-free-log-tau-prior-sigma",
    str(independent_scaling_free_log_tau_prior_sigma),
])

if config.get("softening_length_kpc", 0.0) > 0.0:
    cmd.extend([
        "--softening-length-kpc",
        str(config["softening_length_kpc"]),
        "--softening-length-prior-log-sigma",
        str(config["softening_length_prior_log_sigma"]),
    ])

cmd.extend([
    "--pos-sigma-arcsec",
    "0.1",
    "--image-presence-penalty-weight",
    "2.0",
    "--image-presence-match-radius-arcsec",
    "1.0",
    "--image-presence-temperature-arcsec",
    "0.5",
    "--image-plane-scatter-floor-arcsec",
    "0.01",
    "--scaling-scatter",
    "--exact-image-min-distance-arcsec",
    "0.2",
    "--exact-image-precision-limit",
    "1.0e-4",
    "--image-catalog-family-cutout-image-dir",
    "data/ff_sims",
    "--image-catalog-family-cutout-image-scale",
    "auto",
    "--image-catalog-family-cutout-bands",
    *FF_RGB_BANDS,
    "--image-catalog-family-cutout-rgb-q",
    str(FF_RGB_DISPLAY["q"]),
    "--image-catalog-family-cutout-rgb-stretch",
    str(FF_RGB_DISPLAY["stretch"]),
    "--image-catalog-family-cutout-rgb-minimum",
    str(FF_RGB_DISPLAY["minimum"]),
    "--image-catalog-family-cutout-rgb-red-gain",
    str(FF_RGB_DISPLAY["red_gain"]),
    "--image-catalog-family-cutout-rgb-green-gain",
    str(FF_RGB_DISPLAY["green_gain"]),
    "--image-catalog-family-cutout-rgb-blue-gain",
    str(FF_RGB_DISPLAY["blue_gain"]),
    "--kappa-true-fits",
    config["kappa_true_fits"],
    "--gammax-true-fits",
    config["gammax_true_fits"],
    "--gammay-true-fits",
    config["gammay_true_fits"],
    "--no-jax-clear-caches-after-svi-refresh",
    "--image-catalog-family-cutout-mode",
    "fast",
    "--no-image-catalog-family-cutouts",
    "--numpyro-print-summary",
    "--debug-sampler-diagnostics",
])

print(" ".join(shlex.quote(part) for part in cmd))

In [ ]:
env = os.environ.copy()
env["JAX_NUM_CPU_DEVICES"] = str(cores)

start = time.monotonic()
try:
    subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)
finally:
    print(f"[timing] elapsed={time.monotonic() - start:.2f}s")